# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the `mlcroissant` library in Python. All operations reference entities by their `@id` as defined in the Croissant schema, ensuring reproducibility and interoperability.

### Dataset Source
The dataset is described by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and explore basic dataset information. The metadata contains overall dataset attributes, not the records themselves; records will be loaded in later sections using record set `@id` references.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

In Croissant datasets, data is organized in Record Sets, each identified by a unique `@id`. Let's enumerate all record sets, their fields, field data types, and their respective IDs for reference. All entities are referenced by their `@id`, as recommended for reproducible scientific workflows.

In [ ]:
# List all available record sets with their @id
print("Available Record Sets:")
if not hasattr(metadata, 'recordSets'):
    print("No 'recordSets' found in metadata.\nTrying alternate location...")
    if hasattr(metadata, 'recordSet') and len(metadata.recordSet) > 0:
        record_sets = metadata.recordSet
    else:
        raise RuntimeError("No 'recordSet' found in dataset metadata.")
else:
    record_sets = metadata.recordSets

record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs
    record_set_ids.append(rs_id)
    print(f"  - {rs_id}")

print("\nListing fields per record set:")
for rs_id in record_set_ids:
    try:
        rs_obj = dataset._find_by_id(rs_id)
        print(f"\nRecord Set: {rs_id}")
        if hasattr(rs_obj, 'field'):
            for field in rs_obj.field:
                if isinstance(field, dict) and '@id' in field:
                    field_id = field['@id']
                else:
                    field_id = field
                field_obj = dataset._find_by_id(field_id)
                print(f"    Field: {field_id}")
                print(f"      Name: {getattr(field_obj, 'name', 'N/A')}")
                print(f"      Data type: {getattr(field_obj, 'dataType', 'N/A')}")
        else:
            print("    No fields found for this record set.")
    except Exception as e:
        print(f"    [Could not load fields. Error: {e}]")

## 3. Data Extraction

Now we load the tabular data from the primary record set(s), referencing them by their `@id`. For the FAIR<sup>2</sup> Mast Cell dataset, we expect at least one main record set with MS-based proteomics outputs.

We will reference all discovered record set `@id`s in the code. For example, if the main table is `_:rs_proteins`, that's the `record_set_id` you'll use.

In [ ]:
# Use the previously extracted record_set_ids
# You may uncomment the print below to see what was found in the prior step
# print(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set: {record_set_id}")
    try:
        recs = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} rows. Columns:")
        print(f"    {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load data for {record_set_id} due to error: {e}")

Let's preview the first few rows of the main record set (replace with your target record set `@id`). If there are multiple record sets, pick the one most relevant for MS-based protein data. For demonstration, we'll select the first loaded record set.

In [ ]:
main_record_set_id = record_set_ids[0]
print(f"Preview of main record set: {main_record_set_id}")
df = dataframes[main_record_set_id]
display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field in the main record set and perform basic analysis: filter records, normalize, and group by a key attribute. You may need to refer to the field overview above to pick field `@id`s that correspond to columns like peptide counts, abundance, or similar.

In [ ]:
# Guess a numeric field - try to find one related to total peptide count, abundance, or molecular weight
numeric_candidates = [c for c in df.columns if df[c].dtype.kind in 'fi' or c.lower().startswith('count') or 'abundance' in c.lower()]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Fallback: just use first column
    numeric_field = df.columns[0]
print(f"Using numeric field: {numeric_field}")

# Filter by threshold (e.g., value > 10)
threshold = 10
try:
    filtered_df = df[df[numeric_field] > threshold].copy()
except Exception as e:
    print(f"Could not filter by {numeric_field} due to {e}; using all data.")
    filtered_df = df.copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a categorical field if present
group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or df[c].dtype.name == 'category')]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Let's create some simple plots showing the distribution of the selected numeric field and the grouping (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# Barplot of grouped means if grouping was performed
if 'group_field' in locals():
    plt.figure(figsize=(10, 6))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrated how to load, explore, and analyze a dataset using the `mlcroissant` library, fully referencing all tabular structures and their fields by `@id` as defined in the Croissant schema. You can extend these analyses by selecting other fields, filtering based on additional criteria, or joining across record sets (if available) - always referencing by `@id`, ensuring reproducibility and clarity.